 """Geometry and Viewing Pipeline - Manuel Cantero Ruiz

The program demonstrates:
- Basic 2D transformations using NumPy matrices.
- Image fitting with contain, crop and stretch.
- Mouse-based window selection and random rectangles with a 4:3 ratio.
- A unit cube, perspective projection, viewplane coordinates, clipping,
  and window-to-viewport mapping.

Only NumPy and Matplotlib are required.

Main structure:
- Define example 2D/3D geometry and constants.
- Implement 2D transformations and image/window operations.
- Implement the camera, perspective projection, clipping, and viewport mapping.
- Run mathematical checks or draw examples.
- Launch the interactive GUI, where sliders and mouse events trigger redraws.

"""

In [43]:
from __future__ import annotations

# sys lets the program read command-line options such as --test and --preview.
import sys
# NumPy handles coordinate arrays, matrices, and the synthetic RGB map image.
import numpy as np
# Matplotlib draws the plots and provides the interactive GUI controls.
import matplotlib

In [44]:
# A backend is the system Matplotlib uses to display or save figures.
# Tests and previews only save/check results, so they do not need a GUI window.
if "--test" in sys.argv or "--preview" in sys.argv:
    matplotlib.use("Agg")
else:
    # The GUI needs an interactive backend to receive mouse and slider events.
    # This set contains common backend names that support interaction.
    interactive_backends = {
        "gtk3agg", "gtk3cairo", "gtk4agg", "gtk4cairo", "macosx",
        "qtagg", "qtcairo", "qt5agg", "qt5cairo", "tkagg", "webagg",
        "wx", "wxagg", "wxcairo", "nbagg",
    }
    current_backend = str(matplotlib.get_backend()).lower()
    if not (current_backend in interactive_backends or current_backend.startswith("module://ipympl")):
        # If the current backend cannot interact, try two commonly installed ones.
        # Keep any errors so the final message can explain why switching failed.
        backend_errors = []
        for candidate in ("TkAgg", "QtAgg"):
            try:
                matplotlib.use(candidate, force=True)
                current_backend = str(matplotlib.get_backend()).lower()
                if current_backend in interactive_backends:
                    break
            except Exception as error:
                backend_errors.append(f"{candidate}: {error}")
        if not (current_backend in interactive_backends or current_backend.startswith("module://ipympl")):
            raise SystemExit(
                "An interactive Matplotlib backend is required for mouse selection and sliders. "
                "Run this script from a local terminal and make sure Tkinter or Qt is installed.\n"
                + " | ".join(backend_errors)
            )

# pyplot creates figures and axes; Rectangle draws selection/viewport boxes.
# Slider, Button, and RectangleSelector provide interactive controls.
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.widgets import Slider, Button, RectangleSelector



#
# 1. EXAMPLE GEOMETRY
#

In [45]:

# Each row is one 2D vertex. The last point is joined to the first when drawing.
# This small polygon is used to demonstrate and test 2D transformations.
POLYGON_2D = np.array([
    [1.0, 1.0],
    [3.0, 1.0],
    [2.0, 2.0],
    [1.5, 3.0],
])

# The unit cube has all combinations of x, y and z in {0, 1}.
# The three nested loops produce its eight corner points.
CUBE = np.array([
    [x, y, z]
    for x in (0.0, 1.0)
    for y in (0.0, 1.0)
    for z in (0.0, 1.0)
])

# Join two cube vertices if they differ in exactly one coordinate.
# The sum of absolute coordinate differences is 1 for adjacent cube corners.
CUBE_EDGES = [
    (i, j)
    for i in range(len(CUBE))
    for j in range(i + 1, len(CUBE))
    if np.isclose(np.sum(np.abs(CUBE[i] - CUBE[j])), 1.0)
]

# Camera aims at the centre of the unit cube.
CAMERA_TARGET = np.array([0.5, 0.5, 0.5])
# The viewport is the rectangle inside the screen where projected geometry is drawn.
# Tuple order is (minimum U, minimum V, maximum U, maximum V).
VIEWPORT = (100.0, 100.0, 700.0, 500.0)
SCREEN_WIDTH, SCREEN_HEIGHT = 800.0, 600.0
# Dimensions of the synthetic island image, measured in pixels (rows, columns).
MAP_HEIGHT, MAP_WIDTH = 240, 360
# Horizontal-to-vertical image ratio; also defines the map's coordinate range.
MAP_ASPECT = MAP_WIDTH / MAP_HEIGHT
# A square target makes contain, crop, and stretch look visibly different.
FIT_SIZE = (200, 200)
# Randomly generated selection windows must have this width/height ratio (4:3).
RANDOM_WINDOW_RATIO = 4.0 / 3.0



# 
# 2. BASIC 2D TRANSFORMATIONS
# 


In [46]:
def translation_matrix_2d(tx: float, ty: float) -> np.ndarray:
    """Return a 3x3 homogeneous matrix for translation by (tx, ty)."""
    # Homogeneous coordinates add a third value (1) so translation can be a matrix operation.
    # Row-vector convention: [x, y, 1] @ T = [x + tx, y + ty, 1].
    return np.array([[1.0, 0.0, 0.0],
                     [0.0, 1.0, 0.0],
                     [tx, ty, 1.0]])


def scaling_matrix_2d(sx: float, sy: float) -> np.ndarray:
    """Return a matrix for scaling about the origin."""
    # sx controls horizontal size and sy controls vertical size; 1 means unchanged.
    return np.array([[sx, 0.0, 0.0],
                     [0.0, sy, 0.0],
                     [0.0, 0.0, 1.0]])


def rotation_matrix_2d(angle_degrees: float) -> np.ndarray:
    """Return an anticlockwise 2D rotation matrix; angle is in degrees."""
    # NumPy's trigonometric functions use radians, so convert the input angle first.
    angle = np.deg2rad(angle_degrees)
    # Reuse cosine and sine when building the rotation matrix.
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, s, 0.0],
                     [-s, c, 0.0],
                     [0.0, 0.0, 1.0]])


def reflection_matrix_2d(flip_x: bool = False, flip_y: bool = False) -> np.ndarray:
    """Reflect in the y-axis (flip_x) and/or x-axis (flip_y)."""
    # A reflection is scaling by -1 on the chosen axis and by +1 on the other.
    return scaling_matrix_2d(-1.0 if flip_x else 1.0,
                             -1.0 if flip_y else 1.0)


def shear_matrix_x(factor: float) -> np.ndarray:
    """Return a horizontal shear: x_new = x + factor*y."""
    # Shear slides points horizontally in proportion to their vertical coordinate.
    return np.array([[1.0, 0.0, 0.0],
                     [factor, 1.0, 0.0],
                     [0.0, 0.0, 1.0]])


def apply_2d(points: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """Apply a 3x3 homogeneous transformation matrix to a list of 2D points."""
    # Make sure the input is a NumPy array containing floating-point coordinates.
    points = np.asarray(points, dtype=float)
    # Add W=1 to every [x, y] point so translation can be represented in the matrix.
    points_h = np.column_stack((points, np.ones(len(points))))
    # '@' performs matrix multiplication; each point is transformed by the same matrix.
    transformed_h = points_h @ matrix
    # The third coordinate is homogeneous W. Divide x and y by W to return ordinary coordinates.
    w = transformed_h[:, 2]
    # W=0 cannot be converted to a finite Cartesian point, so raise a clear error.
    if np.any(np.isclose(w, 0.0)):
        raise ValueError("A transformed point has homogeneous coordinate W=0.")
    return transformed_h[:, :2] / w[:, None]


def make_2d_transform(tx: float, ty: float, sx: float, sy: float,
                      angle: float, shear: float) -> np.ndarray:
    """Combine scale, shear, rotation and translation into a single matrix."""
    # Combine several operations into one matrix so every point only needs one matrix product.
    # With row vectors, operations are applied from left to right.
    return (scaling_matrix_2d(sx, sy)
            @ shear_matrix_x(shear)
            @ rotation_matrix_2d(angle)
            @ translation_matrix_2d(tx, ty))



# 
# 3. CREATE A SMALL GAME MAP AND FIT IT INTO A WINDOW
# 


In [47]:
def make_game_map(height: int = MAP_HEIGHT, width: int = MAP_WIDTH) -> np.ndarray:
    """Create a simple island map as an RGB NumPy array (no image library needed)."""
    # Build row/column coordinate grids so each pixel can be assigned a map coordinate.
    rows, cols = np.mgrid[0:height, 0:width]
    # Convert pixel columns to x coordinates from -aspect to +aspect.
    x = (cols / (width - 1) * 2.0 - 1.0) * (width / height)
    # Convert pixel rows to y coordinates from +1 at the top to -1 at the bottom.
    y = 1.0 - rows / (height - 1) * 2.0
    # Store an RGB colour triplet for every pixel; float RGB values range from 0 to 1.
    image = np.zeros((height, width, 3), dtype=float)
    image[:] = [0.40, 0.72, 0.82]  # Sea

    # An ellipse-like equation defines the island's main shape.
    radius = ((x + 0.15) / 1.12) ** 2 + ((y - 0.03) / 0.72) ** 2
    # Small sine/cosine waves make the coast less perfectly smooth.
    coastline = 1.0 + 0.055 * np.sin(7.0 * x) + 0.035 * np.cos(10.0 * y)
    # Boolean masks identify which pixels fall on land and on the sandy edge.
    land = radius < coastline
    sand = land & (radius > coastline - 0.12)
    # Assign green to land and tan to the narrow sandy band around the coast.
    image[land] = [0.34, 0.66, 0.34]
    image[sand] = [0.86, 0.76, 0.51]
    # Add darker hills in the upper-left part of the island using another mask.
    hills = land & (x < -0.25) & (y > 0.05 + 0.12 * np.sin(5.0 * x))
    image[hills] = [0.23, 0.48, 0.27]
    # An ellipse makes a lake; draw it only where it overlaps the island.
    lake = ((x - 0.25) / 0.25) ** 2 + ((y + 0.10) / 0.16) ** 2 < 1.0
    image[lake & land] = [0.24, 0.58, 0.76]
    # Roads are thin bands around two straight-line equations.
    roads = (np.abs(y - (0.30 * x + 0.08)) < 0.018) | (np.abs(y - (-0.55 * x - 0.08)) < 0.014)
    image[land & roads] = [0.93, 0.86, 0.68]
    # Draw three small points of interest at fixed world coordinates.
    for px, py in [(-0.65, -0.05), (0.05, 0.32), (0.58, -0.18)]:
        # A distance-squared test creates a small circular dot around this point.
        dots = (x - px) ** 2 + (y - py) ** 2 < 0.0018
        image[land & dots] = [0.28, 0.20, 0.14]
    # Return the complete (height, width, RGB channels) image array.
    return image


In [48]:
def resize_nearest(image: np.ndarray, new_height: int, new_width: int) -> np.ndarray:
    """Resize an image using nearest-neighbour sampling, using NumPy only."""
    # Read the source image's current number of rows and columns.
    old_height, old_width = image.shape[:2]
    # For each destination row, choose the source row that supplies its colour.
    # Integer conversion selects a nearby source index; minimum prevents an out-of-range index.
    rows = np.minimum((np.arange(new_height) * old_height / new_height).astype(int), old_height - 1)
    # Do the same for columns. Broadcasting rows and columns selects a full destination grid.
    cols = np.minimum((np.arange(new_width) * old_width / new_width).astype(int), old_width - 1)
    # Return a resized image by copying source pixels at the selected row/column indices.
    return image[rows[:, None], cols[None, :]]


In [49]:

def fit_image(image: np.ndarray, target_shape: tuple[int, int], mode: str) -> np.ndarray:
    """Fit an image to (height, width) with 'contain', 'crop' or 'stretch'."""
    # target_shape is (target height, target width), not (width, height).
    target_h, target_w = target_shape
    # Get the original dimensions so we can calculate how to scale the image.
    source_h, source_w = image.shape[:2]
    background = np.array([0.88, 0.88, 0.88])  # Neutral bars make contain visible.

    if mode == "stretch":
        # Stretch fills the window but can change the image proportions.
        # Resize directly to the requested dimensions and finish this function.
        return resize_nearest(image, target_h, target_w)
    if mode not in {"contain", "crop"}:
        raise ValueError("mode must be 'contain', 'crop' or 'stretch'.")

    # Work out the scale needed to match the target width and the target height separately.
    scale_x, scale_y = target_w / source_w, target_h / source_h
    # 'contain' uses the smaller scale so the whole image fits; 'crop' uses the larger scale
    # so the target is fully covered (the extra part will be cut off).
    scale = min(scale_x, scale_y) if mode == "contain" else max(scale_x, scale_y)
    new_w = max(1, int(round(source_w * scale)))
    new_h = max(1, int(round(source_h * scale)))
    # Resize the source while preserving its original width/height proportion.
    resized = resize_nearest(image, new_h, new_w)

    if mode == "contain":
        # Start with a target-sized canvas, filled with the neutral background colour.
        canvas = np.empty((target_h, target_w, 3), dtype=float)
        canvas[:] = background
        # Calculate the top-left placement so the resized image is centred in the canvas.
        x0, y0 = (target_w - new_w) // 2, (target_h - new_h) // 2
        # Put the resized image into the appropriate rectangle of the canvas.
        canvas[y0:y0 + new_h, x0:x0 + new_w] = resized
        return canvas

    # Crop preserves proportions but removes the excess rows or columns.
    # Find the central part of the enlarged image that matches the target dimensions.
    x0, y0 = max(0, (new_w - target_w) // 2), max(0, (new_h - target_h) // 2)
    # Slice out that rectangle; anything outside it is discarded.
    return resized[y0:y0 + target_h, x0:x0 + target_w]



In [50]:

def transform_map(image: np.ndarray, tx: float, ty: float, sx: float, sy: float,
                  angle: float, shear: float) -> np.ndarray:
    """Pan, scale, rotate and shear the map by inverse pixel sampling."""
    # Get image dimensions and its width/height ratio.
    height, width = image.shape[:2]
    aspect = width / height
    # Give every output column an x coordinate and every output row a y coordinate.
    # y runs from top (1) to bottom (-1), matching the order of image rows.
    target_x = np.linspace(-aspect, aspect, width)
    target_y = np.linspace(1.0, -1.0, height)
    # Build a coordinate grid, then flatten it into one [x, y] pair per output pixel.
    xx, yy = np.meshgrid(target_x, target_y)
    output_points = np.column_stack((xx.ravel(), yy.ravel()))
    # Combine pan, scale, rotation and shear into one transformation matrix.
    matrix = make_2d_transform(tx, ty, sx, sy, angle, shear)

    # Inverse mapping: for each destination pixel, find its matching source coordinate.
    # This fills the output pixel grid more reliably than pushing source pixels forward.
    source_points = apply_2d(output_points, np.linalg.inv(matrix))
    # Convert source x coordinates into integer column indices (0 to width - 1).
    col = np.rint((source_points[:, 0] / aspect + 1.0) * 0.5 * (width - 1)).astype(int)
    # Convert source y coordinates into integer row indices; image rows run top-to-bottom.
    row = np.rint((1.0 - source_points[:, 1]) * 0.5 * (height - 1)).astype(int)
    # Mark source coordinates that fall inside the original image's boundaries.
    valid = (col >= 0) & (col < width) & (row >= 0) & (row < height)
    # Create the output image and prefill it with the sea colour for areas outside the source.
    result = np.empty_like(image)
    result[:] = [0.40, 0.72, 0.82]
    # Flatten to one RGB triplet per pixel so the validity mask can select output pixels.
    flat = result.reshape(-1, 3)
    # Copy source colours only for valid locations; invalid locations keep the sea background.
    flat[valid] = image[row[valid], col[valid]]
    # Return the transformed RGB image.
    return result



In [51]:
def crop_selected_window(image: np.ndarray, window: tuple[float, float, float, float]) -> np.ndarray:
    """Crop a selected world-coordinate window from the current map image."""
    # The window is expressed as geometric bounds, not pixel indices.
    xmin, xmax, ymin, ymax = window
    height, width = image.shape[:2]
    aspect = width / height
    # Convert the horizontal world-coordinate bounds into start/end pixel columns.
    col0 = int(np.floor((xmin + aspect) / (2.0 * aspect) * width))
    col1 = int(np.ceil((xmax + aspect) / (2.0 * aspect) * width))
    # Convert the vertical bounds into row indices; image y is reversed (top row first).
    row0 = int(np.floor((1.0 - ymax) / 2.0 * height))
    row1 = int(np.ceil((1.0 - ymin) / 2.0 * height))
    # Clamp every bound so the crop cannot extend beyond the image.
    col0, col1 = np.clip([col0, col1], 0, width)
    row0, row1 = np.clip([row0, row1], 0, height)
    # Ensure the crop has at least one row and one column.
    col1, row1 = max(col0 + 1, col1), max(row0 + 1, row1)
    # Slice the rows and columns to extract the selected rectangular region.
    return image[row0:min(row1, height), col0:min(col1, width)]


In [52]:

def random_window(rng: np.random.Generator, aspect_ratio: float = RANDOM_WINDOW_RATIO,
                  map_aspect: float = MAP_ASPECT) -> tuple[float, float, float, float]:
    """Generate a random rectangle with a prescribed width/height ratio."""
    # Limit height so the rectangle fits entirely inside the map.
    # The rectangle's height must fit both the map's vertical range and horizontal range.
    max_height = min(1.25, 1.9, (2.0 * map_aspect) / aspect_ratio)
    # Choose a random allowed height, then calculate width from the requested ratio.
    height = float(rng.uniform(0.45, max_height))
    width = aspect_ratio * height
    # Choose the bottom-left bounds so the entire rectangle stays inside the map.
    xmin = float(rng.uniform(-map_aspect, map_aspect - width))
    ymin = float(rng.uniform(-1.0, 1.0 - height))
    # Return (minimum x, maximum x, minimum y, maximum y).
    return xmin, xmin + width, ymin, ymin + height


# 
# 4. CAMERA, PERSPECTIVE PROJECTION AND VIEWPLANE COORDINATES
# 


In [53]:

def camera_from_angles(yaw: float, pitch: float, distance: float,
                       target: np.ndarray = CAMERA_TARGET):
    """Return camera position and the forward, right and up unit vectors."""
    # Distance must be positive so the camera is not placed at or through the target.
    if distance <= 0:
        raise ValueError("Camera distance must be positive.")
    # Convert the two angles to radians for the trigonometric functions.
    yaw_r, pitch_r = np.deg2rad([yaw, pitch])
    # Convert yaw/pitch/distance to a 3D offset from the point the camera looks at.
    offset = distance * np.array([
        np.cos(pitch_r) * np.cos(yaw_r),
        np.cos(pitch_r) * np.sin(yaw_r),
        np.sin(pitch_r),
    ])
    # 'eye' is the camera position; 'forward' points from the camera towards the target.
    eye = target + offset
    forward = target - eye
    # Normalize the direction vector so its length is 1.
    forward /= np.linalg.norm(forward)
    # Define the world's vertical direction; cross products derive camera right/up directions.
    world_up = np.array([0.0, 0.0, 1.0])
    # A cross product gives a direction perpendicular to both forward and world-up.
    right = np.cross(forward, world_up)
    right_length = np.linalg.norm(right)
    if right_length < 1e-9:
        raise ValueError("Camera direction is too close to the vertical axis.")
    # Normalize right, then compute and normalize up to form the camera's orthogonal basis.
    right /= right_length
    up = np.cross(right, forward)
    up /= np.linalg.norm(up)
    # Return camera position plus its forward, right, and up unit vectors.
    return eye, forward, right, up


In [54]:
def perspective_projection_matrix(eye: np.ndarray, forward: np.ndarray,
                                  target: np.ndarray = CAMERA_TARGET) -> np.ndarray:
    """Build the 4x4 perspective projection matrix onto the plane through target."""
    # Represent the viewplane with plane coefficients [a, b, c, d].
    # The plane passes through target and has 'forward' as its normal vector.
    plane = np.r_[forward, -np.dot(forward, target)]
    # Add a homogeneous coordinate to the camera position for the projection formula.
    viewpoint = np.r_[eye, 1.0]
    # This dot product is used to build the projective matrix.
    dot_product = float(np.dot(plane, viewpoint))
    if abs(dot_product) < 1e-10:
        raise ValueError("The camera cannot lie on the viewplane.")
    # For a row point P, P @ M is a homogeneous point on the viewplane.
    # The outer product and identity matrix together define the 4x4 projection matrix.
    return np.outer(plane, viewpoint) - dot_product * np.eye(4)


def viewplane_coordinate_matrix(target: np.ndarray, right: np.ndarray,
                                up: np.ndarray) -> np.ndarray:
    """Build VC = K.T @ inverse(K @ K.T), as described in Section 4.4."""
    # K encodes the viewplane's two screen directions and a reference point on the plane.
    # Its rows contain the camera-right vector, camera-up vector, and target position.
    K = np.array([
        [right[0], right[1], right[2], 0.0],
        [up[0], up[1], up[2], 0.0],
        [target[0], target[1], target[2], 1.0],
    ])
    # Build the viewplane-coordinate transform using the formula K.T @ inverse(K @ K.T).
    return K.T @ np.linalg.inv(K @ K.T)


In [55]:

def device_coordinate_matrix(window: tuple[float, float, float, float],
                             viewport: tuple[float, float, float, float] = VIEWPORT) -> np.ndarray:
    """Build DC mapping window=(xmin, xmax, ymin, ymax) to a pixel viewport."""
    # Unpack the geometry window and the pixel viewport boundaries.
    xmin, xmax, ymin, ymax = window
    umin, vmin, umax, vmax = viewport
    # Reject invalid rectangles before dividing by their width/height.
    if xmax <= xmin or ymax <= ymin:
        raise ValueError("Window width and height must be positive.")
    if umax <= umin or vmax <= vmin:
        raise ValueError("Viewport width and height must be positive.")
    # Calculate horizontal and vertical scale factors from window size to viewport size.
    sx = (umax - umin) / (xmax - xmin)
    sy = (vmax - vmin) / (ymax - ymin)
    # The final row adds the translation needed to align the window's lower bounds
    # with the viewport's lower bounds (using row-vector homogeneous coordinates).
    return np.array([
        [sx, 0.0, 0.0],
        [0.0, sy, 0.0],
        [umin - xmin * sx, vmin - ymin * sy, 1.0],
    ])


def to_homogeneous_3d(points: np.ndarray) -> np.ndarray:
    """Append a homogeneous coordinate W=1 to a list of 3D points."""
    # Convert each [x, y, z] point into [x, y, z, 1] for matrix transformations.
    return np.column_stack((np.asarray(points, dtype=float), np.ones(len(points))))


def from_homogeneous(points_h: np.ndarray) -> np.ndarray:
    """Convert homogeneous row points to Cartesian coordinates by dividing by W."""
    # Work with floating-point data and get the final coordinate W from each point.
    points_h = np.asarray(points_h, dtype=float)
    w = points_h[:, -1]
    # W=0 describes a point at infinity; it cannot be represented as finite x/y/z values.
    if np.any(np.isclose(w, 0.0, atol=1e-11)):
        raise ValueError("A point at infinity (W=0) cannot be plotted as finite coordinates.")
    # Divide the remaining coordinates by W to recover ordinary Cartesian coordinates.
    return points_h[:, :-1] / w[:, None]


In [56]:
def clip_segment_liang_barsky(p0: np.ndarray, p1: np.ndarray,
                              window: tuple[float, float, float, float]):
    """Clip one 2D segment against a rectangle; return visible endpoints or None."""
    # Clipping removes the part of a line segment that lies outside the window.
    xmin, xmax, ymin, ymax = window
    # Unpack both endpoints and calculate the segment direction (dx, dy).
    x0, y0 = map(float, p0)
    x1, y1 = map(float, p1)
    dx, dy = x1 - x0, y1 - y0
    # These values describe the four rectangle boundaries: left, right, bottom, top.
    p_values = (-dx, dx, -dy, dy)
    # q_values measure the starting point's position relative to those boundaries.
    q_values = (x0 - xmin, xmax - x0, y0 - ymin, ymax - y0)
    # t=0 is the start of the segment and t=1 is the end; narrow this visible interval.
    t_min, t_max = 0.0, 1.0
    for p_value, q_value in zip(p_values, q_values):
        if abs(p_value) < 1e-14:
            if q_value < 0:
                return None
            continue
        # The ratio says where the segment meets the current boundary.
        ratio = q_value / p_value
        if p_value < 0:
            t_min = max(t_min, ratio)
        else:
            t_max = min(t_max, ratio)
        # If the allowable interval becomes empty, the segment misses the window.
        if t_min > t_max:
            return None
    # Interpolate along the segment to calculate the clipped start and end points.
    start = np.array([x0 + t_min * dx, y0 + t_min * dy])
    end = np.array([x0 + t_max * dx, y0 + t_max * dy])
    return start, end


In [57]:
def build_pipeline(yaw: float, pitch: float, distance: float,
                   zoom: float, pan_x: float, pan_y: float) -> dict:
    """Calculate each pipeline stage and the final combined VP matrix."""
    # First define the camera position and its local orientation axes.
    eye, forward, right, up = camera_from_angles(yaw, pitch, distance)
    # M projects 3D points onto the viewplane; VC converts them to viewplane coordinates.
    M = perspective_projection_matrix(eye, forward)
    VC = viewplane_coordinate_matrix(CAMERA_TARGET, right, up)
    umin, vmin, umax, vmax = VIEWPORT
    # Larger zoom values make the visible window smaller (zoom in).
    window_width = 2.5 / zoom
    # Match the window's aspect ratio to the viewport to avoid unwanted stretching.
    window_height = window_width * (vmax - vmin) / (umax - umin)
    # Define the window around the requested pan centre.
    window = (pan_x - window_width / 2, pan_x + window_width / 2,
              pan_y - window_height / 2, pan_y + window_height / 2)
    # DC maps window coordinates into viewport pixel coordinates.
    DC = device_coordinate_matrix(window)
    # Compose the stages into one matrix: perspective -> viewplane coordinates -> device.
    VP = M @ VC @ DC

    # Add homogeneous W coordinates to the cube's eight vertices.
    cube_h = to_homogeneous_3d(CUBE)
    # Keep each intermediate result so the interface can illustrate every pipeline stage.
    projected_h = cube_h @ M                 # 3D points projected onto the viewplane.
    projected_world = from_homogeneous(projected_h)  # Convert from homogeneous coordinates.
    local_h = projected_h @ VC               # Express points in the viewplane's local axes.
    local = from_homogeneous(local_h)
    device_h = local_h @ DC                  # Map local window coordinates to viewport pixels.
    device = from_homogeneous(device_h)
    # Return both the matrices and intermediate coordinates for drawing and testing.
    return {
        "eye": eye, "forward": forward, "right": right, "up": up,
        "M": M, "VC": VC, "DC": DC, "VP": VP, "window": window,
        "cube_h": cube_h, "projected_h": projected_h,
        "projected_world": projected_world, "local_h": local_h,
        "local": local, "device": device,
    }




# 
# 5. NUMERICAL TESTS (NO GUI REQUIRED)
# 


In [58]:
def run_tests() -> None:
    """Check the important geometric properties before using the interface."""
    # With no movement, unit scale, and zero angles, the combined transform is identity.
    identity = make_2d_transform(0, 0, 1, 1, 0, 0)
    assert np.allclose(apply_2d(POLYGON_2D, identity), POLYGON_2D)
    # Check that applying a translation and then its inverse restores the polygon.
    translation = translation_matrix_2d(2, -1.5)
    moved = apply_2d(POLYGON_2D, translation)
    assert np.allclose(apply_2d(moved, np.linalg.inv(translation)), POLYGON_2D)
    # Reflecting twice in the same axis should restore the original points.
    reflection = reflection_matrix_2d(flip_x=True)
    assert np.allclose(apply_2d(apply_2d(POLYGON_2D, reflection), reflection), POLYGON_2D)

    # Check image size and make sure the three fitting modes produce different results.
    game_map = make_game_map()
    fitted = {mode: fit_image(game_map, (180, 180), mode)
              for mode in ("contain", "crop", "stretch")}
    assert all(image.shape == (180, 180, 3) for image in fitted.values())
    assert not np.allclose(fitted["contain"], fitted["crop"])
    assert not np.allclose(fitted["contain"], fitted["stretch"])

    # A generated rectangle must respect its exact prescribed aspect ratio.
    rect = random_window(np.random.default_rng(15), aspect_ratio=4/3)
    xmin, xmax, ymin, ymax = rect
    assert np.isclose((xmax - xmin) / (ymax - ymin), 4/3)
    assert -MAP_ASPECT <= xmin < xmax <= MAP_ASPECT
    assert -1 <= ymin < ymax <= 1

    # Build a sample camera pipeline for the projection and coordinate-mapping tests.
    pipeline = build_pipeline(-65, 25, 4.2, 1.0, 0.0, 0.0)
    _, forward, right, up = camera_from_angles(-65, 25, 4.2)
    K = np.array([[*right, 0.0], [*up, 0.0], [*CAMERA_TARGET, 1.0]])
    # Verify the VC matrix and check that projected points lie on the viewplane.
    assert np.allclose(K @ pipeline["VC"], np.eye(3))
    assert np.allclose((pipeline["projected_world"] - CAMERA_TARGET) @ forward, 0.0, atol=1e-9)
    # Mapping through each stage should match applying the combined VP matrix once.
    by_stages = pipeline["cube_h"] @ pipeline["M"] @ pipeline["VC"] @ pipeline["DC"]
    at_once = pipeline["cube_h"] @ pipeline["VP"]
    assert np.allclose(by_stages, at_once, atol=1e-9)

    # The two opposite window corners should map to the corresponding viewport corners.
    xmin, xmax, ymin, ymax = pipeline["window"]
    corners = np.array([[xmin, ymin, 1.0], [xmax, ymax, 1.0]]) @ pipeline["DC"]
    assert np.allclose(from_homogeneous(corners), [[100, 100], [700, 500]])
    clipped = clip_segment_liang_barsky(np.array([-2., 0.]), np.array([2., 0.]), (-1, 1, -1, 1))
    assert clipped is not None and np.allclose(clipped, [[-1, 0], [1, 0]])
    assert clip_segment_liang_barsky(np.array([-2., 2.]), np.array([-1.5, 3.]), (-1, 1, -1, 1)) is None

    changed = build_pipeline(15, 20, 4.2, 1.0, 0.0, 0.0)
    assert not np.allclose(pipeline["local"], changed["local"])
    print("All tests passed: 2D transforms, image fitting, prescribed-ratio random windows, camera, projection, VC, clipping, DC and VP.")



# 
# 6. DRAWING HELPERS
# 


In [59]:
def draw_cube(ax, pipeline: dict, yaw: float, pitch: float, distance: float) -> None:
    """Draw the unit cube and the camera position in 3D."""
    # Clear the previous frame before redrawing with the latest camera values.
    ax.clear()
    # Draw each edge using its two endpoint vertices.
    for i, j in CUBE_EDGES:
        ax.plot(*zip(CUBE[i], CUBE[j]), color="tab:blue", linewidth=1.6)
    ax.scatter(CUBE[:, 0], CUBE[:, 1], CUBE[:, 2], color="tab:blue", s=14)
    # Mark the camera and draw a dashed line from the camera to the look-at target.
    eye = pipeline["eye"]
    ax.scatter(*eye, color="tab:red", s=32, label="Camera")
    ax.plot([eye[0], CAMERA_TARGET[0]], [eye[1], CAMERA_TARGET[1]],
            [eye[2], CAMERA_TARGET[2]], "--", color="tab:red", linewidth=1.1)
    ax.scatter(*CAMERA_TARGET, color="black", s=18)
    # Use the cube, camera, and target together to choose plot limits that show everything.
    cloud = np.vstack((CUBE, eye[None, :], CAMERA_TARGET[None, :]))
    low, high = cloud.min(axis=0), cloud.max(axis=0)
    centre = (low + high) / 2
    span = max(float(np.max(high - low)), 1.8)
    ax.set_xlim(centre[0] - span * 0.58, centre[0] + span * 0.58)
    ax.set_ylim(centre[1] - span * 0.58, centre[1] + span * 0.58)
    ax.set_zlim(centre[2] - span * 0.58, centre[2] + span * 0.58)
    ax.set_box_aspect((1, 1, 1))
    ax.set_xlabel("x", labelpad=-2)
    ax.set_ylabel("y", labelpad=-2)
    ax.set_zlabel("z", labelpad=-2)
    ax.set_title(f"Unit cube and camera | yaw={yaw:.0f} deg, pitch={pitch:.0f} deg, d={distance:.1f}", fontsize=9)
    ax.legend(loc="upper left", fontsize=8)


In [60]:
def draw_viewport(ax, pipeline: dict) -> None:
    """Draw clipped projected cube edges in the device viewport."""
    # Clear old geometry before drawing the current pipeline result.
    ax.clear()
    umin, vmin, umax, vmax = VIEWPORT
    # Draw the viewport boundary as a green rectangle in screen-pixel coordinates.
    ax.add_patch(Rectangle((umin, vmin), umax - umin, vmax - vmin,
                           fill=False, edgecolor="tab:green", linewidth=1.8))
    # Process each projected cube edge independently.
    for i, j in CUBE_EDGES:
        # Clip the edge to the visible window; fully outside segments return None.
        segment = clip_segment_liang_barsky(pipeline["local"][i], pipeline["local"][j], pipeline["window"])
        if segment is not None:
            # Add homogeneous W=1 coordinates to the clipped endpoints before device mapping.
            points_h = np.column_stack((np.asarray(segment), np.ones(2)))
            points_device = from_homogeneous(points_h @ pipeline["DC"])
            ax.plot(points_device[:, 0], points_device[:, 1], color="tab:blue", linewidth=1.8)
    ax.set_xlim(0, SCREEN_WIDTH)
    ax.set_ylim(0, SCREEN_HEIGHT)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("U (pixels)")
    ax.set_ylabel("V (pixels)")
    ax.set_title("Clipped geometry mapped to the viewport", fontsize=10)
    ax.grid(True, linestyle=":", alpha=0.3)
    ax.text(400, 565, "Screen: 800 x 600", ha="center", fontsize=8, color="0.35")


In [61]:
def create_preview(filename: str = "geometry_window_mapping_preview.png") -> None:
    """Save a static preview image; this does not test mouse interactions."""
    # Prepare one example of the transformed map and one example camera pipeline.
    game_map = make_game_map()
    transformed = transform_map(game_map, 0.25, 0.0, 1.0, 1.0, 0.0, 0.0)
    pipeline = build_pipeline(-65, 25, 4.2, 1.0, 0.0, 0.0)
    # Create a static four-panel figure: map, fitted crop, 3D camera, and viewport output.
    fig = plt.figure(figsize=(13, 8), dpi=120)
    ax_map = fig.add_axes([0.05, 0.56, 0.42, 0.36])
    ax_fit = fig.add_axes([0.55, 0.56, 0.40, 0.36])
    ax_cube = fig.add_axes([0.05, 0.12, 0.42, 0.34], projection="3d")
    ax_view = fig.add_axes([0.55, 0.12, 0.40, 0.34])
    ax_map.imshow(transformed, extent=(-MAP_ASPECT, MAP_ASPECT, -1, 1), origin="upper")
    ax_map.add_patch(Rectangle((-0.75, -0.5), 1.5, 1.0, fill=False, color="gold", linewidth=2))
    ax_map.set_title("Game map and selected window")
    ax_map.set_aspect("equal")
    ax_fit.imshow(fit_image(crop_selected_window(transformed, (-0.75, 0.75, -0.5, 0.5)), FIT_SIZE, "contain"))
    ax_fit.set_title("Selected region fitted with contain")
    ax_fit.set_xticks([]); ax_fit.set_yticks([])
    draw_cube(ax_cube, pipeline, -65, 25, 4.2)
    draw_viewport(ax_view, pipeline)
    fig.suptitle("Geometry and Window Mapping Demonstrator", fontsize=15)
    # Save the figure to an image file and close it to release plotting resources.
    fig.savefig(filename, bbox_inches="tight")
    plt.close(fig)
    print(f"Static preview saved to {filename}")


# 
# 7. INTERACTIVE GUI: SLIDERS, POINT-AND-CLICK WINDOW, AND RANDOM EXAMPLES
# 


In [62]:

def launch_interface() -> None:
    """Launch the interactive app with a draggable rectangle and fit-mode buttons."""
    # Prepare the source map, a random-number generator, and the starting selection.
    game_map = make_game_map()
    rng = np.random.default_rng()
    initial_window = (-0.75, 0.75, -0.5, 0.5)
    # Shared mutable state records the selected window, fit mode, and latest transformed map.
    state = {"window": initial_window, "fit_mode": "contain", "transformed_map": game_map.copy()}

    fig = plt.figure(figsize=(15, 13))
    try:
        fig.canvas.manager.set_window_title("Geometry and Window Mapping Demonstrator")
    except Exception:
        pass
    fig.suptitle("Geometry and Window Mapping Demonstrator", fontsize=15, y=0.985)

    # Four plots occupy the upper half; controls are placed in two columns below.
    ax_map = fig.add_axes([0.045, 0.615, 0.435, 0.305])
    ax_fit = fig.add_axes([0.545, 0.615, 0.405, 0.305])
    ax_cube = fig.add_axes([0.045, 0.345, 0.435, 0.22], projection="3d")
    ax_view = fig.add_axes([0.545, 0.345, 0.405, 0.22])

    # Initial values for every slider. These are copied so the sliders can update 'values'.
    initial_values = {
        "tx": 0.25, "ty": 0.0, "sx": 1.0, "sy": 1.0, "angle": 0.0, "shear": 0.0,
        "yaw": -65.0, "pitch": 25.0, "distance": 4.2, "zoom": 1.0,
        "pan_x": 0.0, "pan_y": 0.0,
    }
    values = initial_values.copy()

    # Create the map image artist once. Updating its data preserves the selector overlay.
    # 'extent' connects image boundaries to map coordinates; origin='upper' puts row 0 on top.
    map_artist = ax_map.imshow(game_map, extent=(-MAP_ASPECT, MAP_ASPECT, -1, 1), origin="upper")
    ax_map.set_xlim(-MAP_ASPECT, MAP_ASPECT)
    ax_map.set_ylim(-1, 1)
    ax_map.set_aspect("equal", adjustable="box")
    ax_map.set_xlabel("Map X")
    ax_map.set_ylabel("Map Y")
    ax_map.set_title("1. Drag a rectangle to choose a window", fontsize=10)
    ax_map.grid(True, alpha=0.15)

    # The output image always has the same shape, so set_data updates it in place.
    # Crop the starting window and fit it into the target preview panel.
    initial_crop = crop_selected_window(game_map, initial_window)
    fit_artist = ax_fit.imshow(fit_image(initial_crop, FIT_SIZE, state["fit_mode"]))
    ax_fit.set_title("2. Selected region -> square target (contain)", fontsize=10)
    ax_fit.set_xticks([])
    ax_fit.set_yticks([])

    # Store slider widgets by key so callbacks can read and reset them later.
    sliders = {}
    # Each slider specification is (state key, displayed label, minimum, maximum, column).
    slider_specs = [
        ("tx", "Map X", -1.5, 1.5, 0),
        ("ty", "Map Y", -1.0, 1.0, 0),
        ("sx", "Scale X", 0.4, 2.2, 0),
        ("sy", "Scale Y", 0.4, 2.2, 0),
        ("angle", "Rotate", -180.0, 180.0, 0),
        ("shear", "Shear", -1.0, 1.0, 0),
        ("yaw", "Yaw", -180.0, 180.0, 1),
        ("pitch", "Pitch", -5.0, 70.0, 1),
        ("distance", "Camera d", 2.5, 8.0, 1),
        ("zoom", "Window zoom", 0.5, 2.8, 1),
        ("pan_x", "Window X", -1.5, 1.5, 1),
        ("pan_y", "Window Y", -1.5, 1.5, 1),
    ]
    # Vertical positions of six slider rows in the left/right control columns.
    row_y = [0.300, 0.270, 0.240, 0.210, 0.180, 0.150]
    # Place all sliders; 'column' decides which half of the figure contains each one.
    for idx, (key, label, low, high, column) in enumerate(slider_specs):
        row = idx if column == 0 else idx - 6
        left = 0.055 if column == 0 else 0.555
        slider_ax = fig.add_axes([left, row_y[row], 0.39, 0.018])
        # Create the slider with its initial value, then keep a reference to it by key.
        slider = Slider(slider_ax, label, low, high, valinit=values[key])
        slider.label.set_fontsize(8)
        slider.valtext.set_fontsize(8)
        sliders[key] = slider

    def get_values() -> None:
        # Copy the current value from every slider into the shared values dictionary.
        for key, slider in sliders.items():
            values[key] = float(slider.val)

    def refresh(_event=None) -> None:
        # Redraw callback: read controls, recalculate geometry, and update all four panels.
        get_values()
        # Apply the 2D slider values to the original map (not to the previous transformed frame).
        transformed = transform_map(game_map, values["tx"], values["ty"], values["sx"],
                                    values["sy"], values["angle"], values["shear"])
        state["transformed_map"] = transformed
        map_artist.set_data(transformed)
        # Re-crop the selected region after transforming the map, then apply the chosen fit mode.
        current_crop = crop_selected_window(transformed, state["window"])
        fit_artist.set_data(fit_image(current_crop, FIT_SIZE, state["fit_mode"]))
        ax_fit.set_title(
            f"2. Fit mode: {state['fit_mode']} | selected ratio "
            f"{(state['window'][1]-state['window'][0])/(state['window'][3]-state['window'][2]):.2f}:1",
            fontsize=10,
        )
        # Recompute the 3D viewing pipeline from the camera/window slider values.
        pipeline = build_pipeline(values["yaw"], values["pitch"], values["distance"],
                                  values["zoom"], values["pan_x"], values["pan_y"])
        # Refresh the 3D camera diagram and the final clipped viewport drawing.
        draw_cube(ax_cube, pipeline, values["yaw"], values["pitch"], values["distance"])
        draw_viewport(ax_view, pipeline)
        fig.canvas.draw_idle()

    def on_select(click, release) -> None:
        """Use a mouse drag to set the selected rectangle in map coordinates."""
        # Ignore drags that started or ended outside the axes, where data coordinates are unavailable.
        if click.xdata is None or click.ydata is None or release.xdata is None or release.ydata is None:
            return
        # Sort drag endpoints so the rectangle works in any drag direction, then clamp to map bounds.
        xmin = float(np.clip(min(click.xdata, release.xdata), -MAP_ASPECT, MAP_ASPECT))
        xmax = float(np.clip(max(click.xdata, release.xdata), -MAP_ASPECT, MAP_ASPECT))
        ymin = float(np.clip(min(click.ydata, release.ydata), -1.0, 1.0))
        ymax = float(np.clip(max(click.ydata, release.ydata), -1.0, 1.0))
        # Ignore very tiny selections that would produce a practically empty crop.
        if xmax - xmin < 0.05 or ymax - ymin < 0.05:
            return
        # Save the new geometric window and redraw the interface.
        state["window"] = (xmin, xmax, ymin, ymax)
        refresh()

    # RectangleSelector implements the point-and-click (click-and-drag) method.
    # It calls on_select when the user finishes dragging a rectangle on the map.
    selector = RectangleSelector(
        ax_map, on_select, useblit=False, button=[1], minspanx=0.05, minspany=0.05,
        spancoords="data", interactive=True,
        props={"facecolor": "gold", "edgecolor": "black", "alpha": 0.22, "linewidth": 1.5},
    )
    selector.extents = initial_window

    # Whenever any slider changes, call refresh() to update the plots.
    for slider in sliders.values():
        slider.on_changed(refresh)

    # Mode buttons let the user compare three image-fitting policies.
    mode_buttons = []
    # Place one button for each fitting mode beneath the crop preview.
    for x, mode in zip((0.545, 0.635, 0.725), ("contain", "crop", "stretch")):
        button_ax = fig.add_axes([x, 0.075, 0.08, 0.038])
        button = Button(button_ax, mode.capitalize())
        button.on_clicked(lambda _event, m=mode: set_fit_mode(m))
        mode_buttons.append(button)

    def set_fit_mode(mode: str) -> None:
        # Store the selected policy and redraw the fitted preview.
        state["fit_mode"] = mode
        refresh()

    random_ax = fig.add_axes([0.055, 0.075, 0.13, 0.038])
    reset_window_ax = fig.add_axes([0.195, 0.075, 0.13, 0.038])
    reset_all_ax = fig.add_axes([0.335, 0.075, 0.13, 0.038])
    random_button = Button(random_ax, "Random 4:3")
    reset_window_button = Button(reset_window_ax, "Reset window")
    reset_all_button = Button(reset_all_ax, "Reset controls")

    def choose_random_window(_event) -> None:
        # Generate a new rectangle with a known 4:3 aspect ratio and move the selector to it.
        state["window"] = random_window(rng, RANDOM_WINDOW_RATIO)
        selector.extents = state["window"]
        refresh()

    def reset_window(_event) -> None:
        # Restore only the selected rectangle; leave the sliders as they are.
        state["window"] = initial_window
        selector.extents = initial_window
        refresh()

    def reset_controls(_event) -> None:
        # Return every slider to its initial value, and restore the default window and fit mode.
        for slider in sliders.values():
            slider.reset()
        state["fit_mode"] = "contain"
        state["window"] = initial_window
        selector.extents = initial_window
        refresh()

    random_button.on_clicked(choose_random_window)
    reset_window_button.on_clicked(reset_window)
    reset_all_button.on_clicked(reset_controls)
    fig.text(
        0.055, 0.025,
        "Drag on the map to choose a window. Random 4:3 makes a rectangle with a known ratio; "
        "the target image is square to compare contain, crop and stretch.", fontsize=8.5,
    )
    # Draw the initial state once, then open the interactive window and handle user events.
    refresh()
    plt.show(block=True)


In [64]:

def main() -> None:
    """Run tests, save a preview, or launch the interactive application."""
    # Choose behaviour from the command-line arguments; without an option, launch the GUI.
    if "--test" in sys.argv:
        run_tests()
    elif "--preview" in sys.argv:
        create_preview()
    else:
        launch_interface()


# This condition is true when this file is run directly, but not when imported as a module.
if __name__ == "__main__":
    main()